# Phase 3 V1 Balanced and Contextual Rhetorical Role Prediction — Codabench-validated export

This revised notebook keeps the verified V1 checkpoint and exact original tokenizer/model source. It trains a corrected sentence baseline, then compares it with a document-context model. Both start from the saved V1 weights; neither repeats Phase 1 pretraining.

**Submission contract:** the final `submission.zip` contains one root-level `prediction.json` mapping each document ID directly to its ordered list of label strings. The CL/IT grouped diagnostics file is not the submission file. The final cell validates the payload and archive before printing a PASS message.

## Kaggle inputs and what changed

Attach the existing input containing `v1_model.pt` and `bpe_en_25k.json`, plus an input containing `label_vocab.json` and the six CL/IT train, development, and test JSON files. The notebook writes to `/kaggle/working/phase3_rr_v1_balanced_context`.

The reported first run showed falling training loss while CL and IT development scores moved in opposite directions. This version addresses concrete design issues: sentence training gives both domains equal weight; class weights are computed from that balanced training mixture; the random classifier head gets a larger learning rate than the pretrained backbone; and evaluation reports each domain separately, with both fixed 13-label Macro-F1 and a diagnostic over labels represented in each development set. It also compares a document-context model against the fine-tuned sentence baseline, using the same development selection rule. No score is guaranteed; use the untouched test set only for final reporting.

In [ ]:
import os, json, math, random, time, copy, gc, hashlib, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("/kaggle/working/phase3_rr_v1_balanced_context")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SENTENCE_TOKENS = 512
BATCH_SIZE = 4
EPOCHS = 4
PATIENCE = 3
BACKBONE_LR = 1e-5
HEAD_LR = 2e-4
WEIGHT_DECAY = 0.01
HEAD_DROPOUT = 0.15
CLASS_WEIGHT_POWER = 0.5
MAX_CLASS_WEIGHT = 3.0
USE_MIXED_PRECISION = True
NUM_WORKERS = 0
FEATURE_BATCH_SIZE = 4
CONTEXT_EPOCHS = 12
CONTEXT_PATIENCE = 3

print("Device:", DEVICE)
if DEVICE.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Find each supplied asset under Kaggle's mounted input datasets.
INPUT_ROOT = Path("/kaggle/input")
ASSET_OVERRIDES = {
    # Optional explicit paths if Kaggle has more than one file with the same name.
    # Example: "v1_model.pt": "/kaggle/input/my-model/v1_model.pt"
}
REQUIRED_FILES = [
    "bpe_en_25k.json", "v1_model.pt", "label_vocab.json", "CL_train.json", "CL_dev.json",
    "IT_train.json", "IT_dev.json", "CL_test.json", "IT_test.json"
]

def resolve_input_file(filename):
    override = ASSET_OVERRIDES.get(filename)
    if override:
        path = Path(override)
        if not path.is_file(): raise FileNotFoundError(f"Configured asset does not exist: {path}")
        return path
    if not INPUT_ROOT.exists():
        raise FileNotFoundError("/kaggle/input is missing. Run this notebook in Kaggle and attach the project input datasets.")
    hits = sorted(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    if not hits: raise FileNotFoundError(f"Required file {filename!r} was not found under {INPUT_ROOT}.")
    if len(hits) > 1:
        raise RuntimeError(f"Found multiple {filename!r} files. Set ASSET_OVERRIDES[{filename!r}] to the intended Kaggle path: {hits}")
    return hits[0]

ASSETS = {name: resolve_input_file(name) for name in REQUIRED_FILES}
for name, path in ASSETS.items(): print(f"{name}: {path}")

## Original V1 tokenizer loading

The following tokenizer loading and tokenizer implementation cells are copied verbatim from original V1 notebook cells 13 and 14. The original tokenizer uses the `itos`/`merges` payload and word-by-word byte-level BPE.

In [ ]:
#tokenizer from A1
import os
import json

BPE_FILENAME = "bpe_en_25k.json"
bpe_candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    if BPE_FILENAME in files:
        bpe_candidates.append(os.path.join(root, BPE_FILENAME))

if len(bpe_candidates) == 0:
    raise FileNotFoundError(
        f"{BPE_FILENAME} was not found inside /kaggle/input/. "
        "Make sure the BPE-25k dataset is attached."
    )

BPE_PATH = bpe_candidates[0]

print("Tokenizer found:", BPE_PATH)


# Load the vocabulary and learned BPE merges.
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = bpe_data["vocab_size"]

# Make sure we have loaded the tokenizer we actually want.
assert bpe_vocab_size == 25_000
assert len(bpe_itos) == 25_000

print("Vocabulary size:", bpe_vocab_size)
print("Number of merge rules:", len(bpe_merges))


# Token -> ID mapping
bpe_stoi = {
    token: idx
    for idx, token in enumerate(bpe_itos)
}
merge_to_id = {}

for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge

    left_id = bpe_stoi[left_token]
    right_id = bpe_stoi[right_token]

    new_token_id = 261 + merge_index
    merge_to_id[(left_id, right_id)] = new_token_id

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

print("\nSpecial token IDs:")
print("PAD:", PAD_TOKEN_ID)
print("UNK:", UNK_TOKEN_ID)
print("BOS:", BOS_TOKEN_ID)
print("EOS:", EOS_TOKEN_ID)
print("WORD BOUNDARY:", WORD_BOUNDARY_ID)

In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

In [ ]:
# Load and audit the Phase 3 label vocabulary and data schemas.
LABEL_VOCAB_PATH = ASSETS["label_vocab.json"]
with open(LABEL_VOCAB_PATH, "r", encoding="utf-8") as f: label_to_id = json.load(f)
label_to_id = {str(k): int(v) for k, v in label_to_id.items()}
assert len(label_to_id) == 13, f"Expected 13 labels, found {len(label_to_id)}"
assert sorted(label_to_id.values()) == list(range(13)), "label_vocab.json indices must be contiguous 0..12"
id_to_label = {i: label for label, i in label_to_id.items()}
label_set = set(label_to_id)
print("Official label mapping:", label_to_id)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def audit_labeled(path, split_name):
    records = load_json(path)
    if not isinstance(records, list): raise TypeError(f"{split_name}: expected a list of documents")
    ids=[]; sentence_count=0; empty_count=0; counts=Counter()
    for row_num, row in enumerate(records):
        if not isinstance(row, dict): raise TypeError(f"{split_name} row {row_num}: expected object")
        for key in ("id", "text", "labels"):
            if key not in row: raise ValueError(f"{split_name} row {row_num}: missing {key!r}")
        if not isinstance(row["text"], list) or not isinstance(row["labels"], list):
            raise TypeError(f"{split_name} row {row_num}: text and labels must be lists")
        if len(row["text"]) == 0: raise ValueError(f"{split_name} {row['id']}: empty document")
        if len(row["text"]) != len(row["labels"]):
            raise ValueError(f"{split_name} {row['id']}: {len(row['text'])} sentences but {len(row['labels'])} labels")
        ids.append(str(row["id"])); sentence_count += len(row["text"])
        empty_count += sum(not str(x).strip() for x in row["text"])
        unknown=set(row["labels"])-label_set
        if unknown: raise ValueError(f"{split_name} {row['id']}: unknown labels {sorted(unknown)}")
        counts.update(row["labels"])
    if len(ids) != len(set(ids)): raise ValueError(f"{split_name}: duplicate document IDs")
    print(f"{split_name}: {len(records)} documents, {sentence_count} sentences, {empty_count} empty sentences")
    if counts: print("  label counts:", dict(counts))
    return records, set(ids)

CL_TRAIN, cl_train_ids = audit_labeled(ASSETS["CL_train.json"], "CL train")
CL_DEV, cl_dev_ids = audit_labeled(ASSETS["CL_dev.json"], "CL dev")
IT_TRAIN, it_train_ids = audit_labeled(ASSETS["IT_train.json"], "IT train")
IT_DEV, it_dev_ids = audit_labeled(ASSETS["IT_dev.json"], "IT dev")
assert not (cl_train_ids & cl_dev_ids), "CL train/dev document leakage"
assert not (it_train_ids & it_dev_ids), "IT train/dev document leakage"

def document_fingerprints(records):
    return {hashlib.sha256(json.dumps([str(s) for s in row["text"]],ensure_ascii=False,
        separators=(",",":")).encode("utf-8")).hexdigest():str(row["id"]) for row in records}
named_splits={"CL_train":CL_TRAIN,"CL_dev":CL_DEV,"IT_train":IT_TRAIN,"IT_dev":IT_DEV}
fingerprints={name:document_fingerprints(rows) for name,rows in named_splits.items()}
for i,left in enumerate(named_splits):
    for right in list(named_splits)[i+1:]:
        overlap=set(fingerprints[left]) & set(fingerprints[right])
        if overlap: raise ValueError(f"Exact document-text overlap between {left} and {right}: {len(overlap)}")
print("No exact document-text duplicates across the four labeled train/dev splits.")

def load_test_docs(path, domain):
    data = load_json(path)
    if not isinstance(data, dict): raise TypeError(f"{domain} test: expected mapping document ID -> sentence list")
    docs=[]
    for doc_id, sentences in data.items():
        if not isinstance(sentences, list): raise TypeError(f"{domain} test {doc_id}: expected sentence list")
        docs.append((str(doc_id), [str(s) for s in sentences]))
    print(f"{domain} test: {len(docs)} documents, {sum(len(x[1]) for x in docs)} sentences")
    return docs

CL_TEST = load_test_docs(ASSETS["CL_test.json"], "CL")
IT_TEST = load_test_docs(ASSETS["IT_test.json"], "IT")
for domain,docs in (("CL",CL_TEST),("IT",IT_TEST)):
    empty=[doc_id for doc_id,sentences in docs if not sentences]
    if empty: raise ValueError(f"{domain} test contains empty documents: {empty[:5]}")


In [ ]:
# Tokenize examples once. Match the original V1 pretraining convention: no BOS/EOS.
def encode_sentence(sentence):
    ids = tokenizer.encode(str(sentence), add_bos=False, add_eos=False)
    if not ids: ids = [UNK_TOKEN_ID]
    truncated = len(ids) > MAX_SENTENCE_TOKENS
    if truncated: ids = ids[:MAX_SENTENCE_TOKENS]
    return ids, truncated

def make_labeled_examples(records, domain):
    out=[]; n_truncated=0
    for doc in records:
        for sent_idx, (sentence, label) in enumerate(zip(doc["text"], doc["labels"])):
            ids, truncated = encode_sentence(sentence); n_truncated += int(truncated)
            out.append({"input_ids":ids,"label":label_to_id[label],"domain":domain,
                        "doc_id":str(doc["id"]),"sentence_index":sent_idx})
    return out, n_truncated

CL_TRAIN_EX, cl_trunc = make_labeled_examples(CL_TRAIN,"CL")
IT_TRAIN_EX, it_trunc = make_labeled_examples(IT_TRAIN,"IT")
CL_DEV_EX, cl_dev_trunc = make_labeled_examples(CL_DEV,"CL")
IT_DEV_EX, it_dev_trunc = make_labeled_examples(IT_DEV,"IT")
TRAIN_EX = CL_TRAIN_EX + IT_TRAIN_EX
DEV_BY_DOMAIN = {"CL":CL_DEV_EX,"IT":IT_DEV_EX}
print("Sentence examples: train",len(TRAIN_EX),"CL dev",len(CL_DEV_EX),"IT dev",len(IT_DEV_EX))
print("Truncated examples at",MAX_SENTENCE_TOKENS,"tokens:",
      {"CL train":cl_trunc,"IT train":it_trunc,"CL dev":cl_dev_trunc,"IT dev":it_dev_trunc})
print("Training class counts:", {id_to_label[i]:sum(x["label"]==i for x in TRAIN_EX) for i in range(13)})

class RRSentenceDataset(Dataset):
    def __init__(self, examples): self.examples=examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

def collate_labeled(rows):
    seqs=[torch.tensor(r["input_ids"],dtype=torch.long) for r in rows]
    lengths=torch.tensor([len(x) for x in seqs],dtype=torch.long)
    input_ids=pad_sequence(seqs,batch_first=True,padding_value=PAD_TOKEN_ID)
    labels=torch.tensor([r["label"] for r in rows],dtype=torch.long)
    return input_ids,lengths,labels

# Training batches are rebuilt per epoch so CL and IT contribute equal sentence counts.
TRAIN_LOADER=None
DEV_LOADERS={d:DataLoader(RRSentenceDataset(ex),batch_size=BATCH_SIZE,shuffle=False,
                           num_workers=NUM_WORKERS,collate_fn=collate_labeled,pin_memory=DEVICE.type=="cuda")
             for d,ex in DEV_BY_DOMAIN.items()}

## Original V1 architecture

This is original V1 notebook cell 26 copied verbatim. It defines the pretrained V1 module names and parameter shapes. The following Phase 3 cells instantiate this exact class from the checkpoint configuration and load its saved weights strictly.

In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

In [ ]:
# Load the already-trained V1 checkpoint. This does not execute Phase 1 training.
CHECKPOINT_PATH = resolve_input_file("v1_model.pt")
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
    raise ValueError("Expected the supplied V1 checkpoint wrapper with model_state_dict")
model_config = checkpoint.get("config", {})
expected_config = {"vocab_size":25000,"n_embd":768,"n_layer":12,"n_head":12,
                   "n_kv_head":4,"ffn_multiplier":4,"context_length":1024}
for key, expected in expected_config.items():
    if key in model_config and int(model_config[key]) != expected:
        raise ValueError(f"Checkpoint config mismatch for {key}: {model_config[key]} vs {expected}")

backbone = V1LegalLM(
    vocab_size=int(model_config.get("vocab_size",25000)),
    n_embd=int(model_config.get("n_embd",768)),
    n_layer=int(model_config.get("n_layer",12)),
    n_head=int(model_config.get("n_head",12)),
    n_kv_head=int(model_config.get("n_kv_head",4)),
    max_seq_len=int(model_config.get("context_length",1024)),
    ffn_multiplier=int(model_config.get("ffn_multiplier",4)),
    dropout=float(model_config.get("dropout",0.0)))
load_result = backbone.load_state_dict(checkpoint["model_state_dict"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
backbone = backbone.to(DEVICE)
print("Strict V1 checkpoint load succeeded.")
print("Checkpoint:", CHECKPOINT_PATH)
print("Best Phase 1 validation loss:", checkpoint.get("best_val_loss"))
print("Backbone parameters:", f"{sum(p.numel() for p in backbone.parameters()):,}")

In [ ]:
# Reuse the V1 token embedding, Transformer blocks, and final RMSNorm.
# This is the same hidden-state path used inside V1LegalLM.forward, omitting only
# the 25,000-way language-model projection. Verify that equivalence below.
def encode_v1_hidden(v1_model, input_ids):
    x = v1_model.token_embedding(input_ids)
    for block in v1_model.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)
    return v1_model.final_norm(x)

probe_ids=torch.tensor([[4,5,6,7,8]],dtype=torch.long,device=DEVICE)
backbone.eval()
with torch.no_grad():
    full_logits,_=backbone(probe_ids,use_cache=False)
    hidden=encode_v1_hidden(backbone,probe_ids)
    projected_logits=backbone.lm_head(hidden)
assert torch.allclose(full_logits,projected_logits,atol=1e-5,rtol=1e-5)
print("V1 hidden extraction matches original forward path; max difference:",
      float((full_logits-projected_logits).abs().max()))

class V1RhetoricalRolePredictor(nn.Module):
    def __init__(self, pretrained_backbone, n_labels=13, dropout=0.10):
        super().__init__()
        self.backbone=pretrained_backbone
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(self.backbone.n_embd,n_labels)
    def forward(self,input_ids,lengths):
        hidden=encode_v1_hidden(self.backbone,input_ids)
        rows=torch.arange(input_ids.shape[0],device=input_ids.device)
        sentence_state=hidden[rows,lengths.to(input_ids.device)-1]
        return self.classifier(self.dropout(sentence_state))

rr_model=V1RhetoricalRolePredictor(backbone,len(label_to_id),HEAD_DROPOUT).to(DEVICE)
print("RR output classes:",rr_model.classifier.out_features)

In [ ]:
# Report both the fixed official 13-label Macro-F1 and a diagnostic over
# labels represented in the gold development split. The selection metric remains
# fixed-label Macro-F1 so runs stay comparable to the original notebook.
N_LABELS=len(label_to_id)
def metrics_from_ids(gold,pred):
    cm=np.zeros((N_LABELS,N_LABELS),dtype=np.int64)
    for y,p in zip(gold,pred): cm[int(y),int(p)]+=1
    tp=np.diag(cm).astype(float)
    support=cm.sum(axis=1).astype(float)
    predicted=cm.sum(axis=0).astype(float)
    precision=np.divide(tp,predicted,out=np.zeros_like(tp),where=predicted>0)
    recall=np.divide(tp,support,out=np.zeros_like(tp),where=support>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros_like(tp),where=(precision+recall)>0)
    active=support>0
    per_class={id_to_label[i]:{"precision":float(precision[i]),"recall":float(recall[i]),
                              "f1":float(f1[i]),"support":int(support[i]),
                              "predicted_count":int(predicted[i])} for i in range(N_LABELS)}
    return {"macro_precision_all_13":float(precision.mean()),
            "macro_recall_all_13":float(recall.mean()),
            "macro_f1":float(f1.mean()),
            "macro_f1_gold_supported_diagnostic":float(f1[active].mean()) if active.any() else 0.0,
            "gold_supported_label_count":int(active.sum()),
            "per_class":per_class,"confusion_matrix":cm.tolist()}

def amp_context():
    if DEVICE.type!="cuda" or not USE_MIXED_PRECISION: return nullcontext()
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type="cuda",dtype=dtype)

AMP_ENABLED=DEVICE.type=="cuda" and USE_MIXED_PRECISION
AMP_DTYPE=torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported() else torch.float16
SCALER=torch.amp.GradScaler("cuda",enabled=AMP_ENABLED and AMP_DTYPE==torch.float16)

@torch.inference_mode()
def evaluate_loader(loader):
    rr_model.eval(); gold=[]; pred=[]
    for input_ids,lengths,labels in loader:
        input_ids=input_ids.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True)
        with amp_context(): logits=rr_model(input_ids,lengths)
        gold.extend(labels.tolist()); pred.extend(logits.argmax(dim=-1).cpu().tolist())
    return metrics_from_ids(gold,pred)

In [ ]:
# Corrected full-fine-tuning sentence baseline.
# Each epoch uses the same number of CL and IT sentences, sampling the larger
# CL split without replacement and reshuffling the subset on later epochs.
def make_domain_balanced_loader(epoch):
    n=min(len(CL_TRAIN_EX),len(IT_TRAIN_EX))
    rng=np.random.default_rng(SEED+epoch)
    cl_idx=rng.choice(len(CL_TRAIN_EX),size=n,replace=False)
    it_idx=rng.choice(len(IT_TRAIN_EX),size=n,replace=False)
    examples=[CL_TRAIN_EX[int(i)] for i in cl_idx]+[IT_TRAIN_EX[int(i)] for i in it_idx]
    generator=torch.Generator().manual_seed(SEED+epoch)
    return DataLoader(RRSentenceDataset(examples),batch_size=BATCH_SIZE,shuffle=True,
        generator=generator,num_workers=NUM_WORKERS,collate_fn=collate_labeled,
        pin_memory=DEVICE.type=="cuda"),len(examples)

cl_counts=np.bincount([x["label"] for x in CL_TRAIN_EX],minlength=N_LABELS).astype(float)
it_counts=np.bincount([x["label"] for x in IT_TRAIN_EX],minlength=N_LABELS).astype(float)
n_domain=min(len(CL_TRAIN_EX),len(IT_TRAIN_EX))
# Expected label counts in the equal-size CL/IT sampling distribution.
train_counts=(cl_counts/max(len(CL_TRAIN_EX),1)+it_counts/max(len(IT_TRAIN_EX),1))*n_domain
present=train_counts>0
class_weights=np.zeros(N_LABELS,dtype=np.float32)
class_weights[present]=(train_counts[present].mean()/train_counts[present])**CLASS_WEIGHT_POWER
class_weights[present]=np.minimum(class_weights[present],MAX_CLASS_WEIGHT)
CLASS_WEIGHTS=torch.tensor(class_weights,dtype=torch.float32,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
print("Expected counts after equal-domain sampling:",{id_to_label[i]:round(float(train_counts[i]),1) for i in range(N_LABELS)})
print("Smoothed train-only class weights:",{id_to_label[i]:round(float(class_weights[i]),3) for i in range(N_LABELS)})

optimizer=torch.optim.AdamW([
    {"params":rr_model.backbone.parameters(),"lr":BACKBONE_LR},
    {"params":rr_model.classifier.parameters(),"lr":HEAD_LR}],weight_decay=WEIGHT_DECAY)
steps_per_epoch=math.ceil((2*min(len(CL_TRAIN_EX),len(IT_TRAIN_EX)))/BATCH_SIZE)
total_steps=EPOCHS*steps_per_epoch
warmup_steps=max(1,int(0.06*total_steps))
def lr_scale(step):
    if step<warmup_steps: return max(0.1,(step+1)/warmup_steps)
    return max(0.15,(total_steps-step)/max(1,total_steps-warmup_steps))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_scale)

history=[]; best_score=-1.0; best_epoch=None; best_metrics=None; stale_epochs=0
BEST_PATH=OUTPUT_DIR/"best_phase3_v1_rr_sentence.pt"
for epoch in range(1,EPOCHS+1):
    TRAIN_LOADER,n_epoch_examples=make_domain_balanced_loader(epoch)
    rr_model.train(); total_loss=0.0; total_examples=0; start=time.time()
    for input_ids,lengths,labels in TRAIN_LOADER:
        input_ids=input_ids.to(DEVICE,non_blocking=True)
        lengths=lengths.to(DEVICE,non_blocking=True); labels=labels.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            logits=rr_model(input_ids,lengths); loss=criterion(logits,labels)
        SCALER.scale(loss).backward(); SCALER.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(rr_model.parameters(),1.0)
        SCALER.step(optimizer); SCALER.update(); scheduler.step()
        total_loss+=float(loss.detach())*len(labels); total_examples+=len(labels)

    domain_metrics={d:evaluate_loader(DEV_LOADERS[d]) for d in ("CL","IT")}
    selection_score=(domain_metrics["CL"]["macro_f1"]+domain_metrics["IT"]["macro_f1"])/2
    row={"epoch":epoch,"train_loss":total_loss/max(total_examples,1),
         "balanced_train_examples":n_epoch_examples,"dev_CL":domain_metrics["CL"],
         "dev_IT":domain_metrics["IT"],"selection_mean_domain_macro_f1":selection_score,
         "seconds":time.time()-start}
    history.append(row)
    print(f"Epoch {epoch}/{EPOCHS} | train loss {row['train_loss']:.4f} | "
          f"CL Macro-F1 all labels {domain_metrics['CL']['macro_f1']:.4f} "
          f"(gold-supported {domain_metrics['CL']['macro_f1_gold_supported_diagnostic']:.4f}) | "
          f"IT Macro-F1 all labels {domain_metrics['IT']['macro_f1']:.4f} "
          f"(gold-supported {domain_metrics['IT']['macro_f1_gold_supported_diagnostic']:.4f}) | "
          f"mean {selection_score:.4f}")
    if selection_score>best_score:
        best_score=selection_score; best_epoch=epoch; best_metrics=domain_metrics; stale_epochs=0
        torch.save({"backbone_state_dict":rr_model.backbone.state_dict(),
                    "classifier_state_dict":rr_model.classifier.state_dict(),
                    "label_to_id":label_to_id,"id_to_label":id_to_label,
                    "v1_config":model_config,"best_epoch":best_epoch,
                    "dev_metrics":best_metrics,"selection_score":best_score,
                    "class_weights":class_weights.tolist(),
                    "training_config":{"seed":SEED,"max_sentence_tokens":MAX_SENTENCE_TOKENS,
                       "batch_size":BATCH_SIZE,"epochs":EPOCHS,"backbone_lr":BACKBONE_LR,
                       "head_lr":HEAD_LR,"weight_decay":WEIGHT_DECAY,
                       "domain_balanced_sampling":True,"class_weight_power":CLASS_WEIGHT_POWER}},BEST_PATH)
    else:
        stale_epochs+=1
        if stale_epochs>=PATIENCE:
            print("Early stopping: development selection score did not improve."); break

with open(OUTPUT_DIR/"sentence_training_history.json","w",encoding="utf-8") as f:
    json.dump(history,f,ensure_ascii=False,indent=2)
with open(OUTPUT_DIR/"sentence_dev_metrics.json","w",encoding="utf-8") as f:
    json.dump({"best_epoch":best_epoch,"selection_mean_domain_macro_f1":best_score,
               "domains":best_metrics},f,ensure_ascii=False,indent=2)
print("Best sentence-baseline epoch:",best_epoch,"mean domain Macro-F1:",best_score)
print("Saved:",BEST_PATH)

In [ ]:
# Restore the best Phase 3 checkpoint before test inference.
best_state=torch.load(BEST_PATH,map_location=DEVICE,weights_only=True)
rr_model.backbone.load_state_dict(best_state["backbone_state_dict"],strict=True)
rr_model.classifier.load_state_dict(best_state["classifier_state_dict"],strict=True)
rr_model.eval()

def infer_test_domain(domain,docs):
    examples=[]; expected={}
    for doc_id,sentences in docs:
        expected[doc_id]=len(sentences)
        for sent_idx,sentence in enumerate(sentences):
            ids,truncated=encode_sentence(sentence)
            examples.append({"input_ids":ids,"doc_id":doc_id,"sentence_index":sent_idx,"truncated":truncated})
    loader=DataLoader(RRSentenceDataset(examples),batch_size=BATCH_SIZE,shuffle=False,
                      num_workers=NUM_WORKERS,collate_fn=lambda rows: (
                          pad_sequence([torch.tensor(r["input_ids"],dtype=torch.long) for r in rows],
                                       batch_first=True,padding_value=PAD_TOKEN_ID),
                          torch.tensor([len(r["input_ids"]) for r in rows],dtype=torch.long)))
    grouped=defaultdict(list); example_pos=0; n_truncated=0
    with torch.inference_mode():
        for input_ids,lengths in loader:
            input_ids=input_ids.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True)
            with amp_context(): logits=rr_model(input_ids,lengths)
            batch_preds=logits.argmax(dim=-1).cpu().tolist()
            for pred_id in batch_preds:
                meta=examples[example_pos]; example_pos+=1
                grouped[meta["doc_id"]].append((meta["sentence_index"],id_to_label[int(pred_id)]))
                n_truncated+=int(meta["truncated"])
    output=[]
    for doc_id,sentences in docs:
        ordered=[label for _,label in sorted(grouped[doc_id])]
        assert len(ordered)==expected[doc_id]==len(sentences)
        assert all(label in label_set for label in ordered)
        output.append({"id":doc_id,"predictions":ordered})
    assert len(output)==len(docs) and example_pos==sum(expected.values())
    path=OUTPUT_DIR/f"{domain}_test_predictions.json"
    with open(path,"w",encoding="utf-8") as f:
        json.dump({"domain":domain,"documents":output},f,ensure_ascii=False,indent=2)
    # Reload and validate exact coverage and sentence counts.
    saved=load_json(path)["documents"]
    assert [x["id"] for x in saved]==[x[0] for x in docs]
    for row,(_,sentences) in zip(saved,docs):
        assert len(row["predictions"])==len(sentences)
        assert set(row["predictions"])<=label_set
    print(domain,"predicted documents",len(saved),"sentences",example_pos,
          "truncated sentences",n_truncated,"output",path)
    return saved

CL_PREDICTIONS=infer_test_domain("CL",CL_TEST)
IT_PREDICTIONS=infer_test_domain("IT",IT_TEST)
COMBINED_PATH=OUTPUT_DIR/"phase3_rr_predictions.json"
with open(COMBINED_PATH,"w",encoding="utf-8") as f:
    json.dump({"CL":CL_PREDICTIONS,"IT":IT_PREDICTIONS},f,ensure_ascii=False,indent=2)
print("Combined predictions:",COMBINED_PATH)
print("All prediction coverage/order/label checks passed.")

## Context experiment using V1 sentence representations

The sentence-only model treats each sentence independently. This experiment freezes the verified V1 backbone, extracts one vector per sentence, and trains a bidirectional LSTM across each document's sentence sequence. The contextual layer can use preceding and following sentence representations without using any gold labels at inference. This is a separate candidate system; the notebook compares its CL/IT development score with the fine-tuned sentence baseline.

In [ ]:
# Reload the original Phase 1 backbone for frozen feature extraction.
context_backbone=V1LegalLM(
    vocab_size=int(model_config.get("vocab_size",25000)),n_embd=int(model_config.get("n_embd",768)),
    n_layer=int(model_config.get("n_layer",12)),n_head=int(model_config.get("n_head",12)),
    n_kv_head=int(model_config.get("n_kv_head",4)),max_seq_len=int(model_config.get("context_length",1024)),
    ffn_multiplier=int(model_config.get("ffn_multiplier",4)),dropout=float(model_config.get("dropout",0.0)))
context_backbone.load_state_dict(checkpoint["model_state_dict"],strict=True)
context_backbone=context_backbone.to(DEVICE).eval()
for parameter in context_backbone.parameters(): parameter.requires_grad_(False)

def extract_document_features(records,domain,split_name,is_test=False):
    if is_test:
        docs=[{"id":doc_id,"text":sentences} for doc_id,sentences in records]
    else: docs=records
    examples=[]; doc_order=[]; labels_by_doc={}; truncated=0
    for doc in docs:
        doc_id=str(doc["id"]); doc_order.append(doc_id)
        labels_by_doc[doc_id]=doc.get("labels")
        for sent_idx,sentence in enumerate(doc["text"]):
            ids,trunc=encode_sentence(sentence); truncated+=int(trunc)
            examples.append({"input_ids":ids,"doc_id":doc_id,"sentence_index":sent_idx})
    loader=DataLoader(RRSentenceDataset(examples),batch_size=FEATURE_BATCH_SIZE,shuffle=False,
        num_workers=NUM_WORKERS,collate_fn=lambda rows:(
            pad_sequence([torch.tensor(r["input_ids"],dtype=torch.long) for r in rows],
                         batch_first=True,padding_value=PAD_TOKEN_ID),
            torch.tensor([len(r["input_ids"]) for r in rows],dtype=torch.long)))
    grouped=defaultdict(list); pos=0
    with torch.inference_mode():
        for input_ids,lengths in loader:
            input_ids=input_ids.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True)
            with amp_context(): hidden=encode_v1_hidden(context_backbone,input_ids)
            rows=torch.arange(input_ids.shape[0],device=DEVICE)
            vectors=hidden[rows,lengths-1].float().cpu()
            for vector in vectors:
                meta=examples[pos]; pos+=1
                label_list=labels_by_doc[meta["doc_id"]]
                label=None if label_list is None else label_to_id[label_list[meta["sentence_index"]]]
                grouped[meta["doc_id"]].append((meta["sentence_index"],vector,label))
    result=[]
    for doc_id in doc_order:
        rows=sorted(grouped[doc_id],key=lambda x:x[0])
        features=torch.stack([x[1] for x in rows])
        labels=None if rows[0][2] is None else torch.tensor([x[2] for x in rows],dtype=torch.long)
        result.append({"id":doc_id,"domain":domain,"features":features,"labels":labels})
    print(split_name,"documents",len(result),"sentences",pos,"truncated",truncated)
    return result

CTX_CL_TRAIN=extract_document_features(CL_TRAIN,"CL","CL train context")
CTX_IT_TRAIN=extract_document_features(IT_TRAIN,"IT","IT train context")
CTX_CL_DEV=extract_document_features(CL_DEV,"CL","CL dev context")
CTX_IT_DEV=extract_document_features(IT_DEV,"IT","IT dev context")
CTX_CL_TEST=extract_document_features(CL_TEST,"CL","CL test context",is_test=True)
CTX_IT_TEST=extract_document_features(IT_TEST,"IT","IT test context",is_test=True)
del context_backbone; gc.collect()
if DEVICE.type=="cuda": torch.cuda.empty_cache()

In [ ]:
class ContextDocumentDataset(Dataset):
    def __init__(self,documents): self.documents=documents
    def __len__(self): return len(self.documents)
    def __getitem__(self,index): return self.documents[index]

def collate_documents(rows):
    features=pad_sequence([r["features"] for r in rows],batch_first=True,padding_value=0.0)
    lengths=torch.tensor([len(r["features"]) for r in rows],dtype=torch.long)
    labels=None
    if rows[0]["labels"] is not None:
        labels=pad_sequence([r["labels"] for r in rows],batch_first=True,padding_value=-100)
    return features,lengths,labels,[r["id"] for r in rows],[r["domain"] for r in rows]

class V1ContextBiLSTM(nn.Module):
    def __init__(self,input_dim=int(model_config.get("n_embd",768)),hidden_dim=256,n_layers=2,n_labels=13,dropout=0.25):
        super().__init__()
        self.lstm=nn.LSTM(input_dim,hidden_dim,num_layers=n_layers,batch_first=True,
                          bidirectional=True,dropout=dropout if n_layers>1 else 0.0)
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(hidden_dim*2,n_labels)
    def forward(self,features,lengths):
        packed=pack_padded_sequence(features,lengths.cpu(),batch_first=True,enforce_sorted=False)
        packed_out,_=self.lstm(packed)
        sequence,_=pad_packed_sequence(packed_out,batch_first=True,total_length=features.shape[1])
        return self.classifier(self.dropout(sequence))

context_model=V1ContextBiLSTM(n_labels=N_LABELS).to(DEVICE)
ctx_dev_loaders={
    "CL":DataLoader(ContextDocumentDataset(CTX_CL_DEV),batch_size=2,shuffle=False,collate_fn=collate_documents),
    "IT":DataLoader(ContextDocumentDataset(CTX_IT_DEV),batch_size=2,shuffle=False,collate_fn=collate_documents)}
ctx_optimizer=torch.optim.AdamW(context_model.parameters(),lr=1e-3,weight_decay=1e-3)
ctx_criterion=nn.CrossEntropyLoss(weight=CLASS_WEIGHTS,ignore_index=-100,reduction="none")

def paired_context_batches(epoch):
    # One CL and one IT document per update: each domain gets equal update weight.
    n=min(len(CTX_CL_TRAIN),len(CTX_IT_TRAIN))
    if n==0: raise ValueError("Context training requires non-empty CL and IT training splits")
    rng=np.random.default_rng(SEED+1000+epoch)
    cl_idx=rng.choice(len(CTX_CL_TRAIN),size=n,replace=False)
    it_idx=rng.choice(len(CTX_IT_TRAIN),size=n,replace=False)
    order=rng.permutation(n)
    for k in order:
        yield collate_documents([CTX_CL_TRAIN[int(cl_idx[k])],CTX_IT_TRAIN[int(it_idx[k])]])

@torch.inference_mode()
def evaluate_context_loader(loader):
    context_model.eval(); gold=[]; pred=[]
    for features,lengths,labels,doc_ids,domains in loader:
        features=features.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True)
        labels=labels.to(DEVICE,non_blocking=True)
        logits=context_model(features,lengths); mask=labels!=-100
        gold.extend(labels[mask].tolist()); pred.extend(logits.argmax(-1)[mask].cpu().tolist())
    return metrics_from_ids(gold,pred)

def train_context_experiment():
    best=-1.0; best_epoch=None; best_domain_metrics=None; stale=0; hist=[]
    path=OUTPUT_DIR/"best_phase3_v1_context_bilstm.pt"
    for epoch in range(1,CONTEXT_EPOCHS+1):
        context_model.train(); total=0.0; steps=0; start=time.time()
        for features,lengths,labels,doc_ids,domains in paired_context_batches(epoch):
            features=features.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True)
            labels=labels.to(DEVICE,non_blocking=True)
            ctx_optimizer.zero_grad(set_to_none=True)
            logits=context_model(features,lengths)
            token_losses=ctx_criterion(logits.reshape(-1,N_LABELS),labels.reshape(-1)).reshape(labels.shape)
            domain_losses=[]
            for domain in ("CL","IT"):
                doc_mask=torch.tensor([d==domain for d in domains],device=DEVICE).unsqueeze(1)
                mask=doc_mask & labels.ne(-100)
                selected_labels=labels[mask]
                if selected_labels.numel():
                    denom=CLASS_WEIGHTS[selected_labels].sum().clamp_min(1e-12)
                    domain_losses.append(token_losses[mask].sum()/denom)
            if len(domain_losses)!=2: raise RuntimeError("A paired context batch must include both domains")
            loss=torch.stack(domain_losses).mean()
            loss.backward(); torch.nn.utils.clip_grad_norm_(context_model.parameters(),1.0)
            ctx_optimizer.step(); total+=float(loss.detach()); steps+=1
        domain_metrics={d:evaluate_context_loader(ctx_dev_loaders[d]) for d in ("CL","IT")}
        score=(domain_metrics["CL"]["macro_f1"]+domain_metrics["IT"]["macro_f1"])/2
        row={"epoch":epoch,"mean_batch_loss":total/max(steps,1),"dev_CL":domain_metrics["CL"],
             "dev_IT":domain_metrics["IT"],"selection_mean_domain_macro_f1":score,
             "seconds":time.time()-start}; hist.append(row)
        print(f"Context epoch {epoch}/{CONTEXT_EPOCHS} | loss {row['mean_batch_loss']:.4f} | "
              f"CL {domain_metrics['CL']['macro_f1']:.4f} "
              f"(supported {domain_metrics['CL']['macro_f1_gold_supported_diagnostic']:.4f}) | "
              f"IT {domain_metrics['IT']['macro_f1']:.4f} "
              f"(supported {domain_metrics['IT']['macro_f1_gold_supported_diagnostic']:.4f}) | mean {score:.4f}")
        if score>best:
            best=score; best_epoch=epoch; best_domain_metrics=domain_metrics; stale=0
            torch.save({"context_state_dict":context_model.state_dict(),"label_to_id":label_to_id,
                        "id_to_label":id_to_label,"v1_config":model_config,"best_epoch":epoch,
                        "dev_metrics":domain_metrics,"selection_score":score,
                        "training_config":{"architecture":"2-layer bidirectional LSTM over frozen V1 sentence vectors",
                                           "hidden_dim":256,"learning_rate":1e-3,"class_weights":class_weights.tolist()}},path)
        else:
            stale+=1
            if stale>=CONTEXT_PATIENCE:
                print("Context early stopping."); break
    with open(OUTPUT_DIR/"context_training_history.json","w",encoding="utf-8") as f:
        json.dump(hist,f,ensure_ascii=False,indent=2)
    with open(OUTPUT_DIR/"context_dev_metrics.json","w",encoding="utf-8") as f:
        json.dump({"best_epoch":best_epoch,"selection_mean_domain_macro_f1":best,
                   "domains":best_domain_metrics},f,ensure_ascii=False,indent=2)
    return best,best_epoch,best_domain_metrics,path

context_best_score,context_best_epoch,context_best_metrics,CONTEXT_BEST_PATH=train_context_experiment()
ctx_state=torch.load(CONTEXT_BEST_PATH,map_location=DEVICE,weights_only=True)
context_model.load_state_dict(ctx_state["context_state_dict"],strict=True)
context_model.eval()
print("Context best epoch",context_best_epoch,"mean domain Macro-F1",context_best_score)

In [ ]:
@torch.inference_mode()
def predict_context_docs(documents,domain):
    loader=DataLoader(ContextDocumentDataset(documents),batch_size=1,shuffle=False,
                      collate_fn=collate_documents,num_workers=0)
    output=[]
    for features,lengths,_,doc_ids,domains in loader:
        features=features.to(DEVICE); lengths=lengths.to(DEVICE)
        logits=context_model(features,lengths)
        ids=logits.argmax(-1)[0,:int(lengths[0])].cpu().tolist()
        labels=[id_to_label[int(i)] for i in ids]
        assert len(labels)==int(lengths[0]) and set(labels)<=label_set
        output.append({"id":doc_ids[0],"predictions":labels})
    assert len(output)==len(documents)
    path=OUTPUT_DIR/f"{domain}_test_predictions_context.json"
    with open(path,"w",encoding="utf-8") as f:
        json.dump({"domain":domain,"documents":output},f,ensure_ascii=False,indent=2)
    return output

CTX_CL_PREDICTIONS=predict_context_docs(CTX_CL_TEST,"CL")
CTX_IT_PREDICTIONS=predict_context_docs(CTX_IT_TEST,"IT")
assert [r["id"] for r in CTX_CL_PREDICTIONS]==[x[0] for x in CL_TEST]
assert [r["id"] for r in CTX_IT_PREDICTIONS]==[x[0] for x in IT_TEST]

BASELINE_SCORE=best_score
if context_best_score>BASELINE_SCORE:
    selected_kind="context_bilstm"
    final_cl,final_it=CTX_CL_PREDICTIONS,CTX_IT_PREDICTIONS
else:
    selected_kind="balanced_sentence_finetune"
    final_cl,final_it=CL_PREDICTIONS,IT_PREDICTIONS
DOMAIN_PREDICTIONS_PATH=OUTPUT_DIR/"phase3_rr_predictions_by_domain.json"
with open(DOMAIN_PREDICTIONS_PATH,"w",encoding="utf-8") as f:
    json.dump({"CL":final_cl,"IT":final_it},f,ensure_ascii=False,indent=2)

# Codabench submission: prediction.json at the archive root, keyed by original
# document ID, with one sentence-ordered role label per sentence.
expected_lengths={}
for domain_docs in (CL_TEST,IT_TEST):
    for document_id,sentences in domain_docs:
        key=str(document_id)
        if key in expected_lengths: raise ValueError(f"Duplicate test document ID across domains: {key}")
        expected_lengths[key]=len(sentences)
submission_predictions={}
for domain_rows in (final_cl,final_it):
    for row in domain_rows:
        key=str(row["id"]); labels=list(row["predictions"])
        if key in submission_predictions: raise ValueError(f"Duplicate prediction document ID: {key}")
        if key not in expected_lengths: raise ValueError(f"Unexpected prediction document ID: {key}")
        if len(labels)!=expected_lengths[key]:
            raise ValueError(f"{key}: predicted {len(labels)} labels for {expected_lengths[key]} sentences")
        if not set(labels)<=label_set: raise ValueError(f"{key}: prediction contains unknown labels")
        submission_predictions[key]=labels
if set(submission_predictions)!=set(expected_lengths):
    missing=sorted(set(expected_lengths)-set(submission_predictions))
    extra=sorted(set(submission_predictions)-set(expected_lengths))
    raise ValueError(f"Prediction/test ID mismatch; missing={missing[:5]}, extra={extra[:5]}")

PREDICTION_PATH=OUTPUT_DIR/"prediction.json"
with open(PREDICTION_PATH,"w",encoding="utf-8") as f:
    json.dump(submission_predictions,f,ensure_ascii=False,indent=2)

# Independent pre-upload validation of the serialized JSON. This guards against
# accidentally packaging the CL/IT diagnostic structure used by earlier drafts.
def validate_codabench_payload(payload,where):
    if not isinstance(payload,dict):
        raise ValueError(f"{where}: expected a JSON object keyed by document ID")
    actual_ids=set(payload)
    expected_ids=set(expected_lengths)
    if actual_ids!=expected_ids:
        if actual_ids=={"CL","IT"}:
            raise ValueError(f"{where}: found the old domain-wrapped CL/IT format. Codabench expects document IDs at the JSON top level, each mapped directly to a list of label strings.")
        missing=sorted(expected_ids-actual_ids); extra=sorted(actual_ids-expected_ids)
        raise ValueError(f"{where}: document IDs differ from test data; missing={missing[:5]}, extra={extra[:5]}")
    for document_id,roles in payload.items():
        if not isinstance(roles,list): raise ValueError(f"{where}: {document_id} must map to a JSON list of labels")
        if len(roles)!=expected_lengths[document_id]:
            raise ValueError(f"{where}: {document_id} has {len(roles)} labels for {expected_lengths[document_id]} sentences")
        if any(not isinstance(role,str) for role in roles):
            raise ValueError(f"{where}: {document_id} contains a non-string label")
        if not set(roles)<=label_set:
            raise ValueError(f"{where}: {document_id} contains a label outside the official vocabulary")

with open(PREDICTION_PATH,"r",encoding="utf-8") as f:
    serialized_predictions=json.load(f)
validate_codabench_payload(serialized_predictions,"prediction.json")
if serialized_predictions!=submission_predictions:
    raise ValueError("Serialized prediction.json differs from predictions validated in memory")

SUBMISSION_ZIP_PATH=OUTPUT_DIR/"submission.zip"
with zipfile.ZipFile(SUBMISSION_ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(PREDICTION_PATH,arcname="prediction.json")
# Reopen and validate the exact file users upload to Codabench.
with zipfile.ZipFile(SUBMISSION_ZIP_PATH,"r") as archive:
    if archive.namelist()!=["prediction.json"]: raise ValueError(f"Unexpected ZIP members: {archive.namelist()}")
    if archive.testzip() is not None: raise ValueError("Submission ZIP failed CRC validation")
    packaged=json.loads(archive.read("prediction.json").decode("utf-8"))
validate_codabench_payload(packaged,"submission.zip/prediction.json")
if packaged!=serialized_predictions: raise ValueError("Packaged prediction.json differs from validated serialized predictions")

selection={"selected_model":selected_kind,"balanced_sentence_mean_domain_macro_f1":BASELINE_SCORE,
           "context_bilstm_mean_domain_macro_f1":context_best_score,
           "context_bilstm_best_epoch":context_best_epoch,
           "selected_mean_domain_macro_f1":max(BASELINE_SCORE,context_best_score),
           "domain_predictions_file":str(DOMAIN_PREDICTIONS_PATH),
           "codabench_prediction_file":str(PREDICTION_PATH),
           "codabench_submission_zip":str(SUBMISSION_ZIP_PATH),
           "submission_documents":len(submission_predictions),
           "selection_rule":"higher mean of fixed 13-label CL and IT development Macro-F1"}
with open(OUTPUT_DIR/"model_comparison.json","w",encoding="utf-8") as f:
    json.dump(selection,f,ensure_ascii=False,indent=2)
print("Candidate comparison:",selection)
print("Selected domain-grouped predictions:",DOMAIN_PREDICTIONS_PATH)
print("Codabench prediction JSON:",PREDICTION_PATH)
print("Codabench submission ZIP:",SUBMISSION_ZIP_PATH)
print("PASS: flat document-ID JSON schema; root-level prediction.json; ZIP CRC valid.")
print("Validated test documents in submission:",len(packaged))
print("Both candidate prediction sets and per-domain metrics are retained for review.")

## Outputs and interpretation

The notebook writes all artifacts to `/kaggle/working/phase3_rr_v1_balanced_context`: best sentence and context checkpoints, training histories, per-domain development metrics, test diagnostics, and the model comparison. Once inference completes, it also writes `prediction.json` and `submission.zip`. **Upload only `submission.zip` to Codabench.** The ZIP contains exactly one root-level file, `prediction.json`, formatted as `{document_id: [label_for_sentence_1, ...]}`. It is not wrapped in `CL`/`IT` keys and has no per-document `id`/`predictions` objects. The final cell checks the exact test ID set, sentence counts, label types/vocabulary, and ZIP contents; it stops with an error if any check fails.

`phase3_rr_predictions_by_domain.json` is a separate diagnostic with CL/IT grouping. The development sets contain only five documents per domain, and several labels are absent from a domain's gold development split. That makes the model-selection score noisy. The notebook reports the fixed 13-label score and a gold-supported diagnostic; it selects on the fixed 13-label score. Test labels are never used for model selection. `prediction.json` from Phase 1 contains generated text and is not used as rhetorical-role supervision. The submission serialization follows the supplied test data's document-ID-to-sentence-label structure; use any different wrapper only if the Codabench task's official instructions explicitly require one.

## Research references

1. Malik, V., et al. (2022). “Semantic Segmentation of Legal Documents via Rhetorical Roles.” *Proceedings of the Natural Legal Language Processing Workshop*, ACL. Introduces the 13-role legal-document segmentation task and contextual document setting: https://aclanthology.org/2022.nllp-1.13/
2. Jain, D., Borah, M. D., and Biswas, A. (2023). “NITS_Legal at SemEval-2023 Task 6: Rhetorical Roles Prediction of Indian Legal Documents via Sentence Sequence Labeling Approach.” *SemEval 2023*. Uses local sentence context and a BiLSTM sequence-labeling model: https://aclanthology.org/2023.semeval-1.103/
3. Jin, X. and Wang, Y. (2023). “TeamShakespeare at SemEval-2023 Task 6: Understand Legal Documents with Contextualized Large Language Models.” *SemEval 2023*. Models intra- and inter-sentence context for rhetorical-role prediction: https://aclanthology.org/2023.semeval-1.72/
4. Kalamkar, P., et al. (2022). “Corpus for Automatic Structuring of Legal Documents.” *LREC 2022*. Describes the legal judgment corpus and rhetorical-role annotations: https://aclanthology.org/2022.lrec-1.470/
5. scikit-learn developers. `f1_score` documentation, including macro averaging: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html